## Module 1 - Data Sources - Homework

In [12]:
# Dowload data
import yfinance as yf
import requests
# Data processing
import pandas as pd
import datetime as dt
import numpy as np

# Data visualization
import matplotlib.pyplot as plt


In [13]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

# Fetch the HTML content with headers
response = requests.get(url, headers=headers)

In [22]:
sp_500 = pd.read_html(response.text)[0]

In [23]:
sp_500

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


## Which year had the highest number of additions (starting from 2020)?

In [24]:
sp_500 = sp_500[['Symbol','Security', 'Date added']].copy()
# Convert 'Date added' to datetime
sp_500['Date added'] = pd.to_datetime(sp_500['Date added'])
# Extract year from 'Date added'
sp_500['Year'] = sp_500['Date added'].dt.year

In [25]:
sp_500.head()

,Symbol,Security,Date added,Year
0,MMM,3M,1957-03-04,1957
1,AOS,A. O. Smith,2017-07-26,2017
2,ABT,Abbott Laboratories,1957-03-04,1957
3,ABBV,AbbVie,2012-12-31,2012
4,ACN,Accenture,2011-07-06,2011


In [28]:
mask = sp_500['Year']>=2020
sp_500_filtered = sp_500.loc[mask]

In [29]:
sp_500_filtered.head()

,Symbol,Security,Date added,Year
11,ABNB,Airbnb,2023-09-18,2023
37,APO,Apollo Global Management,2024-12-23,2024
40,APP,AppLovin,2025-09-22,2025
42,ACGL,Arch Capital Group,2022-11-01,2022
44,ARES,Ares Management,2025-12-11,2025


In [30]:
sp_500_filtered_groupby_year = sp_500_filtered.groupby('Year').size().sort_values(ascending=False)
sp_500_filtered_groupby_year

Year
2025    18
2024    16
2022    15
2023    15
2026    13
2020    10
2021    10
dtype: int64

In [31]:
# Stocks that have been > 20 years. (2026-20=2006)

sp_500[sp_500['Date added'] < '2006'].count()

Symbol        218
Security      218
Date added    218
Year          218
dtype: int64

## How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?

In [32]:
tickers = ['^GSPC','000001.SS','^HSI','^AXJO','^NSEI','^GSPTSE','^GDAXI','^FTSE','^N225','^MXX','^BVSP']
df = yf.download(tickers, start="2026-01-01", end="2026-08-21")

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  11 of 11 completed


In [33]:
df = df['Close'].copy()

In [34]:
df.head()

Ticker,000001.SS,^AXJO,^BVSP,^FTSE,^GDAXI,^GSPC,^GSPTSE,^HSI,^MXX,^N225,^NSEI
Date,,,,,,,,,,,
2026-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26146.550781
2026-01-02,NaN,8727.799805,160539.0,9951.099609,24539.339844,6858.470215,31883.400391,26338.470703,64141.359375,NaN,26328.550781
2026-01-05,4023.416992,8728.599609,161870.0,10004.599609,24868.689453,6902.049805,32220.000000,26347.240234,65014.371094,51832.800781,26250.300781
2026-01-06,4083.666992,8714.599609,163664.0,10122.700195,24892.199219,6944.819824,32407.000000,26710.449219,65022.238281,52518.078125,26178.699219
2026-01-07,4085.771973,8695.599609,161975.0,10048.200195,25122.259766,6920.930176,32135.500000,26458.949219,64871.699219,51961.980469,26140.750000


In [35]:
# Calculate YTD returns for each index
rets = {}

for col in df.columns:
    
    start = df[col].dropna().iloc[0]
    end = df[col].dropna().iloc[-1]
    ret = round((end/start-1)*100,2)
    rets[col] = ret
    
rets = pd.DataFrame.from_dict(rets, orient='index', columns=['YTD Return'])
rets.index.name = 'Ticker'  
rets.sort_values(by='YTD Return',ascending=False)

,YTD Return
Ticker,
^N225,27.75
^GSPTSE,14.06
^GSPC,11.41
^FTSE,8.01
^GDAXI,5.88
^BVSP,4.60
^AXJO,4.08
^MXX,0.32
^HSI,-2.43


## 2 out of 10 indexes 

Additional: How many of these indexes have better returns than the S&P 500 over 3, 5, and 10 year periods? 

In [36]:
df3 = yf.download(tickers, start="2024-01-01", end="2026-08-21")
df5 = yf.download(tickers, start="2022-01-01", end="2026-08-21")
df10 = yf.download(tickers, start="2017-01-01", end="2026-08-21")

[*********************100%***********************]  11 of 11 completed
[*********************100%***********************]  11 of 11 completed
[*********************100%***********************]  11 of 11 completed


In [37]:
df3 = df3['Close'].copy()
df5 = df5['Close'].copy()
df10 = df10['Close'].copy()

In [40]:
# Calculate YTD returns for each index over 3 years period
rets3 = {}

for col in df3.columns:
    
    start = df3[col].dropna().iloc[0]
    end = df3[col].dropna().iloc[-1]
    ret = round((end/start-1)*100,2)
    rets3[col] = ret
    
rets3 = pd.DataFrame.from_dict(rets3, orient='index', columns=['YTD Return'])
rets3.index.name = 'Ticker'  
rets3.sort_values(by='YTD Return',ascending=False)

,YTD Return
Ticker,
^N225,98.92
^GSPTSE,74.23
^GSPC,61.11
^GDAXI,54.94
^HSI,53.07
^FTSE,39.20
000001.SS,31.78
^BVSP,26.55
^AXJO,19.09


## 2 out of 10 indexes 

In [42]:
# Calculate YTD returns for each index over 5 years period
rets5 = {}

for col in df5.columns:
    
    start = df5[col].dropna().iloc[0]
    end = df5[col].dropna().iloc[-1]
    ret = round((end/start-1)*100,2)
    rets5[col] = ret
    
rets5 = pd.DataFrame.from_dict(rets5, orient='index', columns=['YTD Return'])
rets5.index.name = 'Ticker'  
rets5.sort_values(by='YTD Return',ascending=False)

,YTD Return
Ticker,
^N225,125.98
^GSPTSE,71.24
^GDAXI,62.18
^BVSP,61.59
^GSPC,59.31
^FTSE,43.21
^NSEI,37.48
^MXX,21.55
^AXJO,19.68


## 4 out of 10 indexes 

In [43]:
# Calculate YTD returns for each index over 10 years period
rets10 = {}

for col in df10.columns:
    
    start = df10[col].dropna().iloc[0]
    end = df10[col].dropna().iloc[-1]
    ret = round((end/start-1)*100,2)
    rets10[col] = ret
    
rets10 = pd.DataFrame.from_dict(rets10, orient='index', columns=['YTD Return'])
rets10.index.name = 'Ticker'  
rets10.sort_values(by='YTD Return',ascending=False)

,YTD Return
Ticker,
^GSPC,238.43
^N225,237.94
^NSEI,196.25
^BVSP,181.81
^GSPTSE,136.09
^GDAXI,124.02
^AXJO,58.44
^FTSE,49.74
^MXX,40.82


None

## Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.

In [44]:
# Download S&P history since 1950
sp_500_history = yf.download('^GSPC', start='1950-01-01')

[*********************100%***********************]  1 of 1 completed


In [46]:
sp_500_history = sp_500_history['Close'].copy()

In [47]:
# Identify maximums (all-time-highs) 
sp_500_history['tops'] = sp_500_history.cummax(axis = 0)

In [48]:
tops = sp_500_history['tops']
prices = sp_500_history['^GSPC']
new_highs = tops[tops != tops.shift(1)]

In [49]:
# Identify corrections > 5%

corrections = []

# Iterate over consecutive peaks (pairs)
for i in range(len(new_highs) - 1):
    start_date = new_highs.index[i]
    end_date = new_highs.index[i + 1]
    peak_value = new_highs.iloc[i]
    
    # Period between 2 consecutive peaks
    drawdown_period = prices.loc[start_date:end_date]
    
    # correction bottom 
    min_value = drawdown_period.min()
    min_date = drawdown_period.idxmin()
    
    # Calculate drawdown %
    drawdown_pct = round((peak_value - min_value) / peak_value * 100, 2)
    
    # Correction duration in days (from top to bottom)
    duration = (min_date - start_date).days
    
    # Filter drawdowns > 5%
    if drawdown_pct >= 5:
        corrections.append({
            'Start (peak)': start_date.date(),
            'Duration [days]': duration,
            'Max drawdown %': drawdown_pct,
            'Bottom date': min_date.date()
        })

# Create dataframe
corrections_df = pd.DataFrame(corrections)

In [50]:
corrections_df['Start (peak)'] = pd.to_datetime(corrections_df['Start (peak)'])
corrections_df['Bottom date'] = pd.to_datetime(corrections_df['Bottom date'])

In [51]:
# Top 10 corrections by max drawdown
corrections_df.sort_values('Max drawdown %', ascending = False).head(10)

,Start (peak),Duration [days],Max drawdown %,Bottom date
56,2007-10-09,517,56.78,2009-03-09
54,2000-03-24,929,49.15,2002-10-09
24,1973-01-11,630,48.20,1974-10-03
22,1968-11-29,543,36.06,1970-05-26
65,2020-02-19,33,33.92,2020-03-23
35,1987-08-25,101,33.51,1987-12-04
15,1961-12-12,196,27.97,1962-06-26
27,1980-11-28,622,27.11,1982-08-12
68,2022-01-03,282,25.43,2022-10-12
18,1966-02-09,240,22.18,1966-10-07


In [52]:
corrections_df['Duration [days]'].describe()

count     74.000000
mean     110.324324
std      175.942480
min        7.000000
25%       22.000000
50%       40.500000
75%       86.250000
max      929.000000
Name: Duration [days], dtype: float64

In [53]:
corrections_df['Max drawdown %'].describe()

count    74.000000
mean     12.518514
std      10.898303
min       5.000000
25%       6.232500
50%       7.985000
75%      14.020000
max      56.780000
Name: Max drawdown %, dtype: float64

## Earnings Surprise Analysis for Amazon (AMZN)

In [54]:
ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)
result = ticker_obj.get_earnings_dates()

In [55]:
AMZN = yf.download('AMZN')

[*********************100%***********************]  1 of 1 completed


In [56]:
AMZN = AMZN['Close'].copy()

In [57]:
AMZN

Ticker,AMZN
Date,
1997-05-15,0.097917
1997-05-16,0.086458
1997-05-19,0.085417
1997-05-20,0.081771
1997-05-21,0.071354
...,...
2026-09-04,258.510010
2026-09-08,256.970001
2026-09-09,252.399994


In [67]:
AMZN['2_day_return'] = AMZN['AMZN'].shift(-1) / AMZN['AMZN'].shift(1) - 1

In [59]:
AMZN.head()

Ticker,AMZN,2_day_pct_forward
Date,,
1997-05-15,0.097917,-0.127659
1997-05-16,0.086458,-0.054211
1997-05-19,0.085417,-0.164639
1997-05-20,0.081771,-0.146494
1997-05-21,0.071354,0.051097


In [64]:
earnings = result[['Reported EPS', 'Surprise(%)']].copy()

In [65]:
earnings.index = earnings.index.date

In [66]:
AMZN.index = AMZN.index.date

In [68]:
combined = earnings.join(AMZN[['2_day_return']], how='inner')

In [69]:
combined

,Reported EPS,Surprise(%),2_day_return
2025-05-01,1.59,16.73,0.030149
2025-02-06,1.86,24.47,-0.029724
2024-10-31,1.43,25.17,0.026981
2024-08-01,1.26,22.58,-0.102043
2024-04-30,0.98,17.91,-0.010831
2024-02-01,1.00,24.55,0.107023
2023-10-26,0.94,60.85,0.052311
2023-08-03,0.65,85.73,0.088605
2023-04-27,0.31,46.36,0.004477
2023-02-02,0.25,42.56,-0.016738


In [70]:
combined = combined.dropna(subset=['Surprise(%)', '2_day_return'])
positive_surprise = combined[combined['Surprise(%)'] > 0]


In [71]:
median_return = positive_surprise['2_day_return'].median()

print(median_return)

0.02698073503614551
